In [65]:
import pandas as pd
import numpy as np

# Stationarity Tests

In [66]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.filters.hp_filter import hpfilter
from arch.unitroot import PhillipsPerron

In [67]:
df = pd.read_csv('/Users/kartikmisra/Documents/ECO723 Project/inflation_forecasting_data.csv')
df.set_index('DATE',inplace = True)

In [68]:
df.head()

,CPI,CoreCPI,PCE,CorePCE,IP,UNEM,INC,WORK,HS,GS5,TB3MS,CorePCE_alt_PCEPILFE,SPREAD
DATE,,,,,,,,,,,,,
1984-01-01,102.1,42.19156,2419.4,47.121,53.0985,8.0,0.6,92673,1897,11.37,8.90,47.121,2.47
1984-02-01,102.6,42.31529,2403.5,47.418,53.3318,7.8,-1.3,93157,2260,11.54,9.09,47.418,2.45
1984-03-01,102.9,42.48026,2431.6,47.598,53.5929,7.8,0.8,93429,1663,12.02,9.52,47.598,2.50
1984-04-01,103.3,42.68647,2457.5,47.822,53.9100,7.7,0.7,93792,1851,12.37,9.69,47.822,2.68
1984-05-01,103.5,42.85145,2474.5,47.933,54.2026,7.4,0.6,94098,1774,13.17,9.83,47.933,3.34


In [69]:
PRICE_INDEX_COLS = ["CPI", "CoreCPI", "PCE","CorePCE"]   # -> log-diff inflation
GAP_TRANSFORM_COLS = ["IP", "UNEM", "WORK", "HS"]          # -> HP-filter gap
NO_TRANSFORM_COLS = ["INC", "SPREAD"]                      # left in levels
 
# HP filter smoothing parameter for MONTHLY df (Ravn-Uhlig convention)
HP_LAMBDA = 14400

In [70]:
def to_annualized_inflation(price_series):
    #Y_t = 1200 * [log(P_t) - log(P_t-1)]
    log_p = np.log(price_series)
    return 1200 * log_p.diff()
 
 
def to_hp_gap(series, lamb=HP_LAMBDA):
    """gap = actual - HP filtered trend"""
    series_clean = series.dropna()
    cycle, trend = hpfilter(series_clean, lamb=lamb)
    return cycle  # the "cycle" component IS the gap (actual - trend)


In [71]:
transformed = pd.DataFrame(index=df.index)
 
for col in PRICE_INDEX_COLS:
    if col in df.columns:
        transformed[col] = to_annualized_inflation(df[col])
 
for col in GAP_TRANSFORM_COLS:
    if col in df.columns:
        transformed[col] = to_hp_gap(df[col])
 
for col in NO_TRANSFORM_COLS:
    if col in df.columns:
        transformed[col] = df[col]
 
transformed = transformed.dropna(how="all")



## Unit Root Test

In [72]:
def clean_series(series):
    s = series.replace([np.inf, -np.inf], np.nan).dropna()
    return s

In [73]:
def run_adf(series, regression):
    s = clean_series(series)
    stat, pvalue, usedlag, nobs, crit, icbest = adfuller(
        s, regression=regression, autolag="AIC"
    )
    return stat, pvalue

In [74]:
def run_pp(series, trend):
    s = clean_series(series)
    pp = PhillipsPerron(s, trend=trend)
    return pp.stat, pp.pvalue

In [75]:
def significance_stars(pvalue):
    if pvalue < 0.01:
        return "***"
    elif pvalue < 0.05:
        return "**"
    elif pvalue < 0.10:
        return "*"
    return ""


In [76]:
def run_all_tests(data, label):
    """Run ADF and PP, both with intercept-only and intercept+trend,
    for every column in `data`. Returns a results DataFrame matching
    the layout of the paper's Table 1."""
    rows = []
    for col in data.columns:
        series = data[col]
        if series.dropna().shape[0] < 10:
            print(f"  Skipping {col}: not enough observations after dropna")
            continue

        adf_c_stat, adf_c_p = run_adf(series, "c")
        adf_ct_stat, adf_ct_p = run_adf(series, "ct")
        pp_c_stat, pp_c_p = run_pp(series, "c")
        pp_ct_stat, pp_ct_p = run_pp(series, "ct")

        rows.append({
            "Series": col,
            "ADF (intercept)": f"{adf_c_stat:.2f}{significance_stars(adf_c_p)}",
            "PP (intercept)": f"{pp_c_stat:.2f}{significance_stars(pp_c_p)}",
            "ADF (intercept+trend)": f"{adf_ct_stat:.2f}{significance_stars(adf_ct_p)}",
            "PP (intercept+trend)": f"{pp_ct_stat:.2f}{significance_stars(pp_ct_p)}",
        })

    result_df = pd.DataFrame(rows).set_index("Series")
    print(f"\n=== {label} ===")
    print(result_df)
    return result_df


In [77]:
level_cols = PRICE_INDEX_COLS + GAP_TRANSFORM_COLS + NO_TRANSFORM_COLS
level_cols = [c for c in level_cols if c in df.columns]

levels_results = run_all_tests(df[level_cols], "LEVEL SERIES (Table 1, left half)")
transformed_results = run_all_tests(transformed, "TRANSFORMED SERIES (Table 1, right half)")


=== LEVEL SERIES (Table 1, left half) ===
        ADF (intercept) PP (intercept) ADF (intercept+trend)  \
Series                                                         
CPI               -0.24          -0.23                 -2.15   
CoreCPI           -1.68         -2.81*                 -1.93   
PCE                2.00           2.20                 -1.91   
CorePCE           -2.12         -2.83*                 -2.00   
IP                -0.93          -0.93                 -2.40   
UNEM             -2.72*          -2.18                 -2.75   
WORK              -1.23          -2.00                 -2.39   
HS                -2.10          -2.08                 -2.28   
INC            -3.45***      -25.10***               -3.50**   
SPREAD         -3.92***        -3.31**              -4.03***   

        PP (intercept+trend)  
Series                        
CPI                    -2.54  
CoreCPI                -1.14  
PCE                    -1.89  
CorePCE                -2.17  
IP

In [81]:
levels_results.to_csv("stationarity_levels.csv")
transformed_results.to_csv("stationarity_transformed.csv")
transformed.to_csv("/Users/kartikmisra/Documents/ECO723 Project/transformed_data.csv")